# Ⅰ第2回 演習2-0 役割の練習（5 分）

演習2・3 と同じ進め方を，小さな課題で 1 回練習する．**全員が自分のノートパソコンで，そろって進む**．

1. (0) で自分の役割を選び，やることを確かめる
2. (1) は implementer の説明を聞いてから書く
3. (2) は verifier が全員の出力を見比べる
4. 「✋ 止まる 0」で，4 つの役割の動き方を練習する

## (0) グループと役割の設定

In [1]:
# ===== (0) グループと役割の設定 =====
GROUP_ID = 12                 # ← 自分のグループ番号 (1〜27) に書き換える
MEMBER_ROLE = "verifier"   # implementer / verifier / recorder / presenter のいずれか（3 人・5 人グループの verifier は "verifier"）

# 授業用フォルダ（AI_TD）のルートを import パスに追加する（ノートブックをどこから開いても動く）
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".dlcourse_root").exists())
sys.path.insert(0, str(ROOT))
print("作業フォルダ:", ROOT)

作業フォルダ: /Users/hinata/AI_TD


## (1) ライブラリのインポートと，自分の役割の確認

In [2]:
import time
import torch
from common.device import get_device, synchronize, machine_string
from common.todo import todo_check, checkpoint

ROLE_TASKS = {
    "implementer": "TODO の答えを先に考え，「この行は何をするか」を口で説明する．✋ では答えの根拠をコードの行と出力を指して示す",
    "verifier":    "全員の出力を見比べ，数値が同じか・桁がおかしくないかを確かめる．違えば原因を探す",
    "recorder":    "✋ の答えを，根拠を含めて 1〜2 文にまとめ，全員に伝える",
    "presenter":   "時間を見て，✋ で問いを読み上げ，話し合いを回す．全員がそろったら「次へ」の合図を出す",
}
if MEMBER_ROLE not in ROLE_TASKS:
    raise ValueError(f"MEMBER_ROLE は {list(ROLE_TASKS)} のどれかにする")
device = get_device()
print(f"あなたの役割：{MEMBER_ROLE}")
print(f"やること　　：{ROLE_TASKS[MEMBER_ROLE]}")

あなたの役割：verifier
やること　　：全員の出力を見比べ，数値が同じか・桁がおかしくないかを確かめる．違えば原因を探す


## (2) 練習 TODO：点数から予測クラスを選ぶ（implementer が説明 → 全員が書く）

3 つの文について，モデルが「否定・肯定」の点数を出した．点数が大きいほうを予測クラス（0 = 否定，1 = 肯定）にする．
演習2 の (6) の TODO 2 で使う `torch.max` の練習．

**進め方**：implementer が「この行は何をするか」を口で説明する → 全員が説明を聞いてから自分で書く．

In [ ]:
scores = torch.tensor([[2.0, 0.5],     # 1 文目：否定 2.0，肯定 0.5
                       [0.1, 1.8],     # 2 文目
                       [1.2, 1.3]])    # 3 文目

# ▼ TODO：右の「...」を消して，行ごと（1 = 横方向）に一番大きい値とその位置を求める式を書く
#   torch.max(テンソル, 1) は「最大値, その位置」の 2 つを返す．位置が予測クラスになる
todo_check("TODO")        # 「...」が残っていたら，ここで止める
values, predicted = torch.max(scores, 1)
print("最大値　　　:", values)
print("予測クラス　:", predicted, "  ← 0 = 否定，1 = 肯定")

最大値　　　: tensor([2.0000, 1.8000, 1.3000])
予測クラス　: tensor([0, 1, 1])   ← 0 = 否定，1 = 肯定


## (3) 練習：全員の出力を見比べる（verifier が確かめる）

全員が実行して，出力をグループで見せ合う．**同じになる値と，人によって違う値** があるはず．verifier が確かめる．

In [16]:
print("予測クラス（全員同じはず）:", predicted.tolist())

a = torch.randn(1024, 1024, device=device)
synchronize(device)
t0 = time.perf_counter()
for _ in range(10):
    b = a @ a
synchronize(device)
print(f"行列積 10 回の時間（人によって違うはず）: {(time.perf_counter() - t0) * 1000:.1f} ms")
print("マシン:", machine_string())

予測クラス（全員同じはず）: [0, 1, 1]
行列積 10 回の時間（人によって違うはず）: 23.1 ms
マシン: Apple M3/Mac15,12/16GB


## ✋ 止まる 0：ここから先へは，グループ全員がそろってから進む

**問い**：予測クラスは全員同じだったか．時間は全員同じだったか．違ったものは何が原因か

1. **presenter**：問いを読み上げ，話し合いを回す
2. **implementer**：答えの根拠を，コードの行と出力を指して説明する（「この行がこう計算したので，この数字になった」）．
   根拠になる所：(2) の `torch.max` の行と予測クラスの出力，(3) の時間とマシンの出力．最後に，次に書く TODO が何をする行かを予告する
3. **verifier**：全員の出力の数値が同じか，桁がおかしくないかを確かめる
4. **recorder**：根拠を含めて答えを 1〜2 文にまとめ，全員に伝える
5. 全員が下のセルの `CP0 = ""` の `" "` の間に，その答えを日本語で書いて実行する．**10 文字より短い（空のままも含む）とエラーになって止まる**

In [17]:
# 問い：予測クラスは全員同じだったか．時間は全員同じだったか．違ったものは何が原因か
# ▼ グループで決めた答えを，下の " " の中に日本語で書く（1〜2 文）
#   書き方の例：CP0 = "〇〇だから，△△になると考えた．"
#   注意：CP0 に入れた文字列が 10 文字より短い（空のままも含む）と，エラーになって先へ進めない
CP0 = "使っている回線が違うから、時間は全員違ったものになると考えた。"
checkpoint(CP0, "止まる 0")

止まる 0 OK：使っている回線が違うから、時間は全員違ったものになると考えた。


## 練習おわり

「✋」の答えを書いて「止まる 0 OK」と出たら終わり．演習2（`ex2.ipynb`）も同じ進め方で進める．
役割を確かめたいときは，(1) の出力を見る．